In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 65 — GitHub Issue Triage Agent

## Goal
Build an agent that **classifies GitHub issues**, infers severity,
**suggests labels**, and **drafts a response** — simulating an
automated triage bot.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Issue classification** | Bug vs feature vs question vs docs |
| **Severity inference** | Critical / high / medium / low |
| **Label suggestion** | Match issue to predefined labels |
| **Response drafting** | Generate helpful first-response |
| **MCP concept** | Tool-based API integration pattern |

## Architecture
```
Issue text → Agent → [classify, infer_severity, suggest_labels, draft_response]
                          ↓
                 Triage Report (type, severity, labels, response)
```

## Stack
- `LangGraph` + `ChatOllama`
- Custom tools simulating GitHub API integration
- Structured output (JSON triage report)

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Define the Label Taxonomy & Sample Issues

A real GitHub project has labels for type, priority, and area.
Our agent needs to know these to suggest the right ones.

In [ ]:
LABEL_TAXONOMY = {
    "type": ["bug", "feature-request", "question", "documentation", "enhancement"],
    "severity": ["critical", "high", "medium", "low"],
    "area": ["frontend", "backend", "api", "database", "auth", "devops", "docs", "testing"],
    "status": ["needs-triage", "confirmed", "needs-info", "wont-fix", "duplicate"],
}

SAMPLE_ISSUES = [
    {
        "id": 1234,
        "title": "App crashes when uploading files larger than 100MB",
        "body": """Steps to reproduce:
1. Go to Upload page
2. Select a file > 100MB
3. Click Upload
4. App shows white screen and console has 'RangeError: Maximum call stack size exceeded'

Expected: File should upload or show error message
Actual: App crashes completely

Browser: Chrome 120, OS: Windows 11
This is blocking our team from uploading training datasets.""",
        "author": "user_alice",
    },
    {
        "id": 1235,
        "title": "Add dark mode support",
        "body": """It would be great to have a dark mode option in the settings.
Many of us work late and the bright UI is hard on the eyes.
Maybe follow the system preference by default?""",
        "author": "user_bob",
    },
    {
        "id": 1236,
        "title": "How do I use the batch API?",
        "body": """I see there's a batch endpoint in the docs but I can't figure out
how to format the request body. The example in the docs returns 400.
Can someone share a working curl example?""",
        "author": "user_carol",
    },
    {
        "id": 1237,
        "title": "Security: SQL injection in search endpoint",
        "body": """The /api/search endpoint is vulnerable to SQL injection.
Sending `q='; DROP TABLE users; --` in the query string causes a 500 error
with a raw SQL error in the response body.

This needs immediate attention.""",
        "author": "user_security_team",
    },
    {
        "id": 1238,
        "title": "Typo in README getting started section",
        "body": """In the README, the installation command says `pip install myapp`
but the correct package name is `pip install my-app` (with a hyphen).
This confused me for 10 minutes.""",
        "author": "user_dave",
    },
]

print(f"Label taxonomy: {json.dumps(LABEL_TAXONOMY, indent=2)}")
print(f"\nSample issues: {len(SAMPLE_ISSUES)}")

## Step 2 — Define Triage Tools

These tools simulate what a real GitHub integration would provide.

In [ ]:
_issues_db = {issue["id"]: issue for issue in SAMPLE_ISSUES}
_triage_results = {}

@tool
def get_issue(issue_id: int) -> str:
    """Fetch a GitHub issue by ID. Returns title, body, and author."""
    if issue_id not in _issues_db:
        return f"Issue #{issue_id} not found."
    issue = _issues_db[issue_id]
    return json.dumps(issue, indent=2)

@tool
def get_available_labels() -> str:
    """Get all available labels organized by category."""
    return json.dumps(LABEL_TAXONOMY, indent=2)

@tool
def search_similar_issues(keywords: str) -> str:
    """Search for existing issues with similar keywords to detect duplicates."""
    matches = []
    for issue in SAMPLE_ISSUES:
        text = (issue["title"] + " " + issue["body"]).lower()
        if any(kw.lower() in text for kw in keywords.split()):
            matches.append(f"#{issue['id']}: {issue['title']}")
    return "\n".join(matches) if matches else "No similar issues found."

@tool
def save_triage_report(issue_id: int, report_json: str) -> str:
    """Save a triage report for an issue. report_json should have keys: type, severity, labels, summary, draft_response."""
    try:
        report = json.loads(report_json)
        required = ["type", "severity", "labels", "summary", "draft_response"]
        missing = [k for k in required if k not in report]
        if missing:
            return f"Missing fields: {missing}"
        _triage_results[issue_id] = report
        return f"✅ Triage report saved for #{issue_id}"
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"

triage_tools = [get_issue, get_available_labels, search_similar_issues, save_triage_report]
print(f"Defined {len(triage_tools)} triage tools")

## Step 3 — Create the Triage Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

TRIAGE_PROMPT = """You are a GitHub issue triage bot. For each issue, you must:

1. Fetch the issue using get_issue()
2. Check available labels with get_available_labels()
3. Search for similar issues to check for duplicates
4. Analyze the issue and create a triage report with:
   - type: bug, feature-request, question, documentation, or enhancement
   - severity: critical, high, medium, or low
   - labels: list of applicable labels from taxonomy
   - summary: one-sentence summary of the issue
   - draft_response: helpful first response to the reporter
5. Save the report using save_triage_report()

Severity guidelines:
- critical: security vulnerabilities, data loss, system crashes affecting all users
- high: major features broken, blocking workflows
- medium: bugs with workarounds, minor feature gaps
- low: cosmetic issues, typos, nice-to-haves

Draft response should be professional, empathetic, and actionable."""

triage_agent = create_react_agent(
    model=llm,
    tools=triage_tools,
    prompt=TRIAGE_PROMPT,
)

def triage_issue(issue_id: int):
    print(f"\n{'='*70}")
    print(f"TRIAGING ISSUE #{issue_id}")
    print(f"{'='*70}")
    result = triage_agent.invoke({
        "messages": [{"role": "user", "content": f"Please triage issue #{issue_id}"}]
    })
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:100]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:300]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 AGENT:\n{msg.content}")
    return result

print("Triage agent ready!")

In [ ]:
# Triage Issue #1234 (bug: file upload crash)
triage_issue(1234)

In [ ]:
# Triage Issue #1237 (security: SQL injection)
triage_issue(1237)

In [ ]:
# Triage Issue #1236 (question: batch API)
triage_issue(1236)

In [ ]:
# Review all triage reports
print("\n" + "="*70)
print("TRIAGE REPORTS SUMMARY")
print("="*70)
for issue_id, report in _triage_results.items():
    print(f"\n#{issue_id}:")
    print(f"  Type:     {report.get('type', 'N/A')}")
    print(f"  Severity: {report.get('severity', 'N/A')}")
    print(f"  Labels:   {report.get('labels', [])}")
    print(f"  Summary:  {report.get('summary', 'N/A')[:100]}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Issue classification** | Agent reads issue and assigns type |
| **Severity inference** | Guidelines-based severity scoring |
| **Duplicate detection** | `search_similar_issues()` finds related issues |
| **Structured output** | Triage report saved as JSON |
| **MCP pattern** | Tools represent API endpoints (GitHub API) |

## MCP Connection (Concept)
```
In production, these tools would connect to real GitHub API via MCP:

MCP Server (GitHub)
  ├── get_issue(owner, repo, number)
  ├── add_labels(issue_number, labels)
  ├── add_comment(issue_number, body)
  └── assign_issue(issue_number, assignee)

The agent calls these tools via the MCP protocol,
which standardizes how LLMs interact with external APIs.
```

## Production Enhancements
- Connect to real GitHub API (or use MCP server)
- Add webhook trigger for new issues
- Integrate with team notification (Slack, email)
- Track triage accuracy over time